In [ ]:
# ============================================================
# EXPERIMENT 6A: NAIVE BAYES CLASSIFICATION FROM SCRATCH
# Student Admission Prediction
# ============================================================
import numpy as np
import pandas as pd

# Step 1: Create Dataset
df = pd.DataFrame({
    "Study_Hours": ["Low", "Low", "Medium", "Medium", "High", "High", "High", "Medium", "Low", "High"],
    "Attendance": ["Low", "High", "Medium", "High", "Medium", "High", "High", "Medium", "Low", "Medium"],
    "Previous_Marks": ["Low", "Medium", "Medium", "High", "High", "High", "Medium", "High", "Low", "High"],
    "Admission": ["No", "No", "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No", "Yes"]
})
print(f"Student Admission Dataset:\n{df}")

# Step 2 & 3: Features, Target, and Prior Probabilities
features = ["Study_Hours", "Attendance", "Previous_Marks"]
target = "Admission"
classes = df[target].unique()
priors = df[target].value_counts(normalize=True).to_dict()

print("\nPrior Probabilities:\n----------------------------------")
for cls, prob in priors.items():
    print(f"{cls} : {round(prob, 4)}")

# Step 4 & 5: Conditional Probability (Laplace Smoothing) and Display
def cond_prob(feature, value, cls):
    class_data = df[df[target] == cls]
    count_feat = len(class_data[class_data[feature] == value])
    return (count_feat + 1) / (len(class_data) + df[feature].nunique())

print("\nConditional Probabilities:\n=============================")
for cls in classes:
    print(f"\nClass: {cls}")
    for feat in features:
        for val in df[feat].unique():
            print(f"P({feat}={val} | Admission={cls}) = {cond_prob(feat, val, cls):.4f}")

# Step 6 & 7: Naive Bayes Prediction Function and Predict New Student
def predict_nb(sample):
    probs = {cls: priors[cls] * np.prod([cond_prob(f, sample[f], cls) for f in features]) for cls in classes}
    return max(probs, key=probs.get), probs

new_student = {"Study_Hours": "High", "Attendance": "High", "Previous_Marks": "High"}
prediction, probabilities = predict_nb(new_student)

# Steps 8-10: Display Prediction and Normalized Probabilities
print("\n================================\nNEW STUDENT\n================================")
print(f"Study Hours : {new_student['Study_Hours']}\nAttendance : {new_student['Attendance']}\nPrevious Marks : {new_student['Previous_Marks']}")

print("\nClass Probabilities:")
for cls, prob in probabilities.items():
    print(f"{cls} : {round(prob, 6)}")

total_prob = sum(probabilities.values())
print("\n=================================================\nNAIVE BAYES PREDICTION\n=================================================")
print(f"Predicted Admission : {prediction}\n\nNormalized Probabilities:")
for cls, prob in probabilities.items():
    print(f"{cls} : {round(prob / total_prob, 4)}")

# Step 11 & 12: Multiple Predictions and Display Results
test_students = pd.DataFrame({
    "Study_Hours": ["High", "Low", "Medium", "High"],
    "Attendance": ["High", "Low", "Medium", "Medium"],
    "Previous_Marks": ["High", "Low", "Medium", "High"]
})
test_students["Predicted_Admission"] = [predict_nb(r)[0] for r in test_students.to_dict('records')]

print("\n=========================================\nMULTIPLE STUDENT PREDICTIONS\n=========================================")
print(test_students)

In [ ]:
# ============================================================
# EXPERIMENT 6B: GAUSSIAN NAIVE BAYES USING SCIKIT-LEARN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
iris = load_iris()
X, y = iris.data, iris.target
feature_names, target_names = iris.feature_names, iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"], df["Species"] = y, df["Target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

print(f"Dataset Shape:\n{df.shape}\n\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{feature_names}\n\nTarget Classes:\n{target_names}\n\nClass Distribution:\n{df['Species'].value_counts()}")

# Step 4-5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-9: Model Training, Prediction, and Results
gnb = GaussianNB().fit(X_train, y_train)
y_pred = gnb.predict(X_test)

results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
results["Actual Species"] = results["Actual"].map(dict(enumerate(target_names)))
results["Predicted Species"] = results["Predicted"].map(dict(enumerate(target_names)))
print(f"\nActual vs Predicted:\n{results}")

# Step 10-12: Performance Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"\n=================================\nGAUSSIAN NAIVE BAYES PERFORMANCE\n=================================")
print(f"Accuracy : {round(accuracy, 4)}\nAccuracy (%) : {round(accuracy * 100, 2)}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}\n\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Step 13: Confusion Matrix Plot
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("GaussianNB Confusion Matrix")
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
plt.xticks(range(len(target_names)), target_names), plt.yticks(range(len(target_names)), target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(), plt.tight_layout(), plt.show()